# Reviewed Gold full training on NVIDIA DGX Spark

This notebook runs one model and one seed at a time. The first authorized run is Qwen2-VL-2B on all 24,107 reviewed training rows (23,499 original plus 608 RETRY/ABORT supplement rows). Checkpoint selection uses all 7,861 original-Gold validation rows. The 194 supplement validation rows are reported separately and cannot select a checkpoint. The locked test split is excluded from download and is never read.

The run saves a rolling `last.ckpt` every registered optimizer-step interval and after every epoch. Rerunning with the same persistent workspace resumes the exact next physical batch, including optimizer, scheduler, scaler, RNG, early-stopping and partial-epoch state. Results are written to per-epoch CSV, source-validation CSV, diagnostics JSON and a complete report JSON.

The reviewed v2.8 mini completed epochs 0-3. Epoch 3 passed all eight registered quality gates; epoch 4 was interrupted by the Kaggle T4 quota. The full-run contract records this compute-limit deviation and does not claim a formal five-epoch mini completion.

The DGX execution profile keeps the accepted mini pixel budget (`50,176` to `200,704`), preserves effective batch size 32 as physical batch 16 x accumulation 2, saves `last.ckpt` every 50 optimizer steps, and runs a fail-closed 16-row profile smoke before full training. It writes to a new run directory so it cannot mix with the earlier high-resolution attempt.

In [ ]:
# 1. Edit only this configuration cell before the first lab run.
from pathlib import Path
import os

USE_EXISTING_LOCAL_REPOSITORY = True
LOCAL_REPOSITORY_ROOT = Path('/home/aiub/kiyas/webagent')
REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPOSITORY_BRANCH = 'Code'
WORKSPACE_ROOT = Path('/home/aiub/kiyas/webagent_full')  # persistent, writable storage

DATA_SOURCE = 'local'  # 'local' for Kaggle-downloaded files; 'huggingface' for snapshots
LOCAL_ORIGINAL_SEARCH_ROOT = WORKSPACE_ROOT / 'data' / 'original'
LOCAL_SUPPLEMENT_SEARCH_ROOT = WORKSPACE_ROOT / 'data' / 'supplement'

# These four values are required only when DATA_SOURCE == 'huggingface'.
ORIGINAL_DATASET_REPO = 'REPLACE_ME/web-gold-40k'
ORIGINAL_DATASET_REVISION = 'REPLACE_WITH_HF_COMMIT_SHA'
SUPPLEMENT_DATASET_REPO = 'REPLACE_ME/gold-40k-retry'
SUPPLEMENT_DATASET_REVISION = 'REPLACE_WITH_HF_COMMIT_SHA'
HF_TOKEN = os.environ.get('HF_TOKEN')  # private dataset token; never paste it here

ACTIVE_MODEL_ID = 'qwen2vl_2b_gold_v2_8_dgx'
SEED = 42
MAX_EPOCHS = 15
MIN_PIXELS = 50_176
MAX_PIXELS = 200_704
PHYSICAL_BATCH_SIZE = 16
GRAD_ACCUM = 2
EFFECTIVE_BATCH_SIZE = 32
CHECKPOINT_EVERY_STEPS = 50
NUM_WORKERS = 8
AUTO_RESUME = True

# The team reported a completed two-person review with no corrections/exclusions.
TWO_PERSON_NO_CHANGE_REVIEW_CONFIRMED = True
REVIEWER_COUNT = 2

# Optional artifact backup after a completed run. Leave disabled during training.
UPLOAD_COMPLETED_RUN_TO_HF = False
MODEL_ARTIFACT_REPO = 'REPLACE_ME/webagent-qwen2vl-2b-gold-v2-8'

MODEL_REGISTRY = {
    'qwen2vl_2b_gold_v2_8_dgx': {
        'label': 'Qwen2-VL-2B Gold v2.8 DGX profile',
        'config': 'configs/backbones/qwen2vl_2b_gold_v2_8_dgx.yaml',
        'full_authorized': True,
        'authorization': (
            'same accepted v2.8 model, losses, pixel budget and effective batch; '
            'GB10 physical-batch execution profile requires its own smoke'
        ),
    },
    'qwen2vl_2b_gold_v2_8': {
        'label': 'Qwen2-VL-2B Gold v2.8',
        'config': 'configs/backbones/qwen2vl_2b_gold_v2_8.yaml',
        'full_authorized': True,
        'authorization': 'completed mini epochs 0-3; epoch 3 passed all eight gates',
    },
    'qwen25vl_3b_gold_v2_8': {
        'label': 'Qwen2.5-VL-3B Gold v2.8',
        'config': 'configs/backbones/qwen25vl_3b_gold_v2_8.yaml',
        'full_authorized': False,
        'authorization': 'blocked until this model completes its own smoke and mini',
    },
    'y1_siglip_roberta': {
        'label': 'Y1 SigLIP + RoBERTa', 'config': 'configs/backbones/y1_siglip_roberta.yaml',
        'full_authorized': False, 'authorization': 'dual-encoder model/fusion path is NotImplemented',
    },
    'y3_clip_roberta': {
        'label': 'Y3 CLIP + RoBERTa', 'config': None,
        'full_authorized': False, 'authorization': 'backbone config and dual-encoder implementation are absent',
    },
    'y7_florence_roberta': {
        'label': 'Y7 Florence-2 + RoBERTa', 'config': None,
        'full_authorized': False, 'authorization': 'encoder/fusion implementation is absent',
    },
    'y2_siglip_large_roberta_large': {
        'label': 'Y2 SigLIP-large + RoBERTa-large', 'config': None,
        'full_authorized': False, 'authorization': 'backbone config and dual-encoder implementation are absent',
    },
    'y6_internvl2_2b': {
        'label': 'Y6 InternVL2-2B', 'config': None,
        'full_authorized': False, 'authorization': 'processor and causal-stream compatibility are unvalidated',
    },
    'a1_to_a5_ablations': {
        'label': 'A1-A5 ablations', 'config': None,
        'full_authorized': False, 'authorization': 'controlled ablation implementations and minis are absent',
    },
    'b1_to_b4_baselines': {
        'label': 'B1-B4 baselines', 'config': None,
        'full_authorized': False, 'authorization': 'separate baseline runners are incomplete',
    },
}

assert ACTIVE_MODEL_ID in MODEL_REGISTRY
assert DATA_SOURCE in {'local', 'huggingface'}
assert WORKSPACE_ROOT != Path('/tmp'), 'Use persistent storage, not /tmp.'
if USE_EXISTING_LOCAL_REPOSITORY:
    assert (LOCAL_REPOSITORY_ROOT / '.git').is_dir(), LOCAL_REPOSITORY_ROOT
if DATA_SOURCE == 'local':
    assert LOCAL_ORIGINAL_SEARCH_ROOT.is_dir(), LOCAL_ORIGINAL_SEARCH_ROOT
    assert LOCAL_SUPPLEMENT_SEARCH_ROOT.is_dir(), LOCAL_SUPPLEMENT_SEARCH_ROOT
else:
    assert 'REPLACE_ME' not in ORIGINAL_DATASET_REPO
    assert 'REPLACE_ME' not in SUPPLEMENT_DATASET_REPO
    assert ORIGINAL_DATASET_REVISION not in {'main', 'REPLACE_WITH_HF_COMMIT_SHA', ''}
    assert SUPPLEMENT_DATASET_REVISION not in {'main', 'REPLACE_WITH_HF_COMMIT_SHA', ''}
    assert HF_TOKEN, 'Export HF_TOKEN for private Hugging Face datasets.'
if UPLOAD_COMPLETED_RUN_TO_HF:
    assert HF_TOKEN, 'Export HF_TOKEN before artifact upload.'
    assert 'REPLACE_ME' not in MODEL_ARTIFACT_REPO


In [ ]:
# 2. Verify CUDA, precision support and persistent storage before network/data work.
import json, platform, shutil, subprocess, sys
from datetime import datetime, timezone
import torch

assert torch.cuda.is_available(), 'A CUDA GPU is required.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_PROPERTIES = torch.cuda.get_device_properties(0)
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage(WORKSPACE_ROOT)
environment_preflight = {
    'status': 'PASS',
    'gpu': GPU_NAME,
    'compute_capability': f'{GPU_PROPERTIES.major}.{GPU_PROPERTIES.minor}',
    'gpu_total_memory_gb': GPU_PROPERTIES.total_memory / 1e9,
    'bf16_supported': torch.cuda.is_bf16_supported(),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'workspace': str(WORKSPACE_ROOT.resolve()),
    'workspace_free_gb': disk.free / 1e9,
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
}
assert environment_preflight['workspace_free_gb'] >= 25, (
    'At least 25 GB free persistent storage is required for data, cache and checkpoints.'
)
print(json.dumps(environment_preflight, indent=2))


In [ ]:
# 3. Freeze one repository checkout and install the runtime. Existing checkouts are not pulled during resume.
import importlib, importlib.metadata as metadata

REPO_ROOT = (
    LOCAL_REPOSITORY_ROOT.resolve()
    if USE_EXISTING_LOCAL_REPOSITORY
    else (WORKSPACE_ROOT / 'repo').resolve()
)
SOURCE_ROOT = REPO_ROOT / 'src'
if USE_EXISTING_LOCAL_REPOSITORY:
    assert (REPO_ROOT / '.git').is_dir(), f'Local repository not found: {REPO_ROOT}'
    print('Using existing local checkout:', REPO_ROOT)
elif not (REPO_ROOT / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', REPOSITORY_BRANCH, '--single-branch',
        REPOSITORY, str(REPO_ROOT),
    ], check=True)
else:
    print('Using frozen existing checkout:', REPO_ROOT)

requirements = [
    'transformers>=4.49,<5', 'peft>=0.14,<1', 'bitsandbytes>=0.45,<1',
    'accelerate>=1,<2', 'scikit-learn>=1.4,<2',
    'huggingface_hub>=0.27,<1', 'pandas>=2,<3', 'pyyaml>=6,<7',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(REPO_ROOT)], check=True)
for module_name in list(sys.modules):
    if module_name == 'web_agent' or module_name.startswith('web_agent.'):
        del sys.modules[module_name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.environ['PYTHONPATH'] = str(SOURCE_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['HF_HOME'] = str(WORKSPACE_ROOT / 'hf_cache')
importlib.invalidate_caches()
os.chdir(REPO_ROOT)
import web_agent
assert Path(web_agent.__file__).resolve().is_relative_to(SOURCE_ROOT.resolve())
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
runtime_environment = {
    **environment_preflight,
    'git_commit': CODE_COMMIT,
    **{name: metadata.version(name) for name in [
        'torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate',
        'scikit-learn', 'huggingface_hub', 'pandas',
    ]},
}
ENVIRONMENT_PATH = WORKSPACE_ROOT / 'environment.json'
ENVIRONMENT_PATH.write_text(json.dumps(runtime_environment, indent=2), encoding='utf-8')
print(json.dumps(runtime_environment, indent=2))


In [ ]:
# 4. Resolve either extracted local Kaggle data or immutable Hugging Face snapshots.
from web_agent.data.recovery_supplement import resolve_supplement_root

def find_original_root(search_root: Path) -> Path:
    candidates = []
    for train_json in search_root.rglob('split_train.json'):
        candidate = train_json.parent.resolve()
        if (candidate / 'split_val.json').is_file() and (candidate / 'images').is_dir():
            candidates.append(candidate)
    candidates = sorted(set(candidates))
    archives = sorted(search_root.rglob('*.zip'))
    assert len(candidates) == 1, (
        'Expected exactly one extracted original root containing split_train.json, '
        f'split_val.json and images/. Found roots: {candidates}; ZIP files: {archives}. '
        'Extract the Kaggle archive before training.'
    )
    return candidates[0]

if DATA_SOURCE == 'huggingface':
    from huggingface_hub import snapshot_download

    DATA_DOWNLOAD_ROOT = WORKSPACE_ROOT / 'data'
    ORIGINAL_SEARCH_ROOT = Path(snapshot_download(
        repo_id=ORIGINAL_DATASET_REPO, repo_type='dataset',
        revision=ORIGINAL_DATASET_REVISION, token=HF_TOKEN,
        local_dir=DATA_DOWNLOAD_ROOT / 'original_gold',
        ignore_patterns=['split_test.json', '**/split_test.json'],
    ))
    SUPPLEMENT_SEARCH_ROOT = Path(snapshot_download(
        repo_id=SUPPLEMENT_DATASET_REPO, repo_type='dataset',
        revision=SUPPLEMENT_DATASET_REVISION, token=HF_TOKEN,
        local_dir=DATA_DOWNLOAD_ROOT / 'retry_abort_supplement_v2',
    ))
    DATASET_SOURCE_IDENTITY = {
        'type': 'huggingface',
        'original': {'repo': ORIGINAL_DATASET_REPO, 'revision': ORIGINAL_DATASET_REVISION},
        'supplement': {'repo': SUPPLEMENT_DATASET_REPO, 'revision': SUPPLEMENT_DATASET_REVISION},
    }
else:
    ORIGINAL_SEARCH_ROOT = LOCAL_ORIGINAL_SEARCH_ROOT.resolve()
    SUPPLEMENT_SEARCH_ROOT = LOCAL_SUPPLEMENT_SEARCH_ROOT.resolve()
    DATASET_SOURCE_IDENTITY = {
        'type': 'local_kaggle_download',
        'original_search_root': str(ORIGINAL_SEARCH_ROOT),
        'supplement_search_root': str(SUPPLEMENT_SEARCH_ROOT),
    }

ORIGINAL_ROOT = find_original_root(ORIGINAL_SEARCH_ROOT)
SUPPLEMENT_ROOT = resolve_supplement_root(SUPPLEMENT_SEARCH_ROOT)
local_test_jsons = sorted(ORIGINAL_SEARCH_ROOT.rglob('split_test.json'))
if local_test_jsons:
    print('Locked test JSON is present locally but will not be opened:', local_test_jsons)
print('ORIGINAL_ROOT =', ORIGINAL_ROOT)
print('SUPPLEMENT_ROOT =', SUPPLEMENT_ROOT)
print('DATASET_SOURCE =', json.dumps(DATASET_SOURCE_IDENTITY, indent=2))


In [ ]:
# 5. Revalidate the accepted data sources and record the manual no-change review attestation.
assert TWO_PERSON_NO_CHANGE_REVIEW_CONFIRMED is True
assert REVIEWER_COUNT == 2
PREFLIGHT_DIR = WORKSPACE_ROOT / 'preflight'
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)
SUPPLEMENT_VALIDATION = PREFLIGHT_DIR / 'supplement_validation.json'
MULTISOURCE_AUDIT = PREFLIGHT_DIR / 'multisource_audit.json'
REVIEW_ATTESTATION = PREFLIGHT_DIR / 'review_attestation.json'
review_attestation = {
    'status': 'PASS',
    'review_mode': 'two_person_manual_no_change_attestation',
    'reviewers': REVIEWER_COUNT,
    'owner_confirmed_no_corrections': True,
    'owner_confirmed_no_exclusions': True,
    'source_records_mutated': False,
    'test_rows_read': 0,
    'limitation': (
        'Manual no-change confirmation has no row-level reconciliation ledger; '
        'report it as an attestation, not an artifact-verified overlay.'
    ),
}
REVIEW_ATTESTATION.write_text(json.dumps(review_attestation, indent=2), encoding='utf-8')
commands = [
    [sys.executable, 'scripts/validate_retry_abort_supplement.py',
     '--supplement-root', str(SUPPLEMENT_ROOT), '--report', str(SUPPLEMENT_VALIDATION)],
    [sys.executable, 'scripts/audit_retry_abort_multisource.py',
     '--original-root', str(ORIGINAL_ROOT), '--supplement-root', str(SUPPLEMENT_ROOT),
     '--report', str(MULTISOURCE_AUDIT)],
]
for command in commands:
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
for report_path in (SUPPLEMENT_VALIDATION, MULTISOURCE_AUDIT):
    gate = json.loads(report_path.read_text(encoding='utf-8'))
    assert gate['status'] == 'PASS', f'Data gate failed: {report_path}'
    assert gate['test_rows_read'] == 0
multisource = json.loads(MULTISOURCE_AUDIT.read_text(encoding='utf-8'))
assert multisource['primary_training_rows'] == 24_107
assert multisource['primary_validation_rows'] == 7_861
assert multisource['supplement_validation_rows_available_separately'] == 194
print('DATA GATES PASSED: 24,107 train / 7,861 original val / 194 supplement val / 0 test reads')


In [ ]:
# 6. Freeze the model/run contract. This cell blocks unverified model backbones.
from web_agent.config import load_config
from web_agent.train.selection import quality_checks

model_spec = MODEL_REGISTRY[ACTIVE_MODEL_ID]
assert model_spec['full_authorized'] is True, (
    f"{model_spec['label']} full run is blocked: {model_spec['authorization']}"
)
CONFIG_PATH = model_spec['config']
cfg = load_config(CONFIG_PATH)
cfg['data']['root'] = str(ORIGINAL_ROOT)
cfg['data']['recovery_supplement']['root'] = str(SUPPLEMENT_ROOT)
cfg['data']['recovery_supplement']['include_in_primary_validation'] = False
cfg['data']['num_workers'] = NUM_WORKERS
cfg['train']['checkpoint_every_steps'] = CHECKPOINT_EVERY_STEPS
assert cfg['backbone']['min_pixels'] == MIN_PIXELS
assert cfg['backbone']['max_pixels'] == MAX_PIXELS
assert cfg['optim']['batch_size'] == PHYSICAL_BATCH_SIZE
assert cfg['optim']['grad_accum'] == GRAD_ACCUM
assert PHYSICAL_BATCH_SIZE * GRAD_ACCUM == EFFECTIVE_BATCH_SIZE
assert cfg['train']['checkpoint_every_steps'] == CHECKPOINT_EVERY_STEPS
assert environment_preflight['gpu_total_memory_gb'] >= 32, (
    'The DGX batch-16 profile requires at least 32 GB GPU/unified memory.'
)
assert cfg['data']['causal_routing'] is True
assert cfg['data']['recovery_transitions'] is True
assert cfg['loss']['hierarchical_recovery'] is True
assert cfg['train']['quality_selection_rule'] == 'all_gates_then_outcome_mcc'

mini_epoch3 = {
    'outcome_mcc': 0.5389, 'action_acc': 0.3740,
    'needs_recovery_macro_f1': 0.5490,
    'needs_recovery_majority_macro_f1': 0.4286,
    'strategy_attempted_macro_f1': 0.4209,
    'strategy_attempted_majority_macro_f1': 0.2393,
    'recovery_outcome_mcc': 0.7631, 'bbox_mean_iou': 0.0698,
    'bbox_recall_iou50': 0.0705, 'outcome_ece': 0.1467,
}
mini_checks = quality_checks(mini_epoch3)
assert all(mini_checks.values()), f'Accepted epoch-3 evidence failed: {mini_checks}'

RUN_DIR = WORKSPACE_ROOT / 'outputs' / ACTIVE_MODEL_ID / f'seed_{SEED}'
CHECKPOINT_ROOT = RUN_DIR / 'checkpoints'
MODEL_CHECKPOINT_DIR = CHECKPOINT_ROOT / f"{cfg['name']}_FULL_SEED{SEED}"
LAST_CHECKPOINT = MODEL_CHECKPOINT_DIR / 'last.ckpt'
METRICS_CSV = RUN_DIR / 'epoch_metrics.csv'
DIAGNOSTICS_JSON = RUN_DIR / 'diagnostics.json'
FULL_REPORT_JSON = RUN_DIR / 'full_report.json'
SOURCE_VALIDATION_CSV = RUN_DIR / 'source_validation.csv'
CONTRACT_PATH = RUN_DIR / 'run_contract.json'
RUN_DIR.mkdir(parents=True, exist_ok=True)
contract = {
    'stage': 'full', 'model_id': ACTIVE_MODEL_ID, 'config': CONFIG_PATH,
    'git_commit': CODE_COMMIT, 'seed': SEED, 'max_epochs': MAX_EPOCHS,
    'checkpoint_every_steps': CHECKPOINT_EVERY_STEPS,
    'execution_profile': cfg['train']['execution_profile'],
    'min_pixels': MIN_PIXELS, 'max_pixels': MAX_PIXELS,
    'physical_batch_size': PHYSICAL_BATCH_SIZE,
    'grad_accum': GRAD_ACCUM, 'effective_batch_size': EFFECTIVE_BATCH_SIZE,
    'mixed_precision': cfg['optim']['mixed_precision'],
    'dataset_source': DATASET_SOURCE_IDENTITY,
    'resolved_original_root': str(ORIGINAL_ROOT),
    'resolved_supplement_root': str(SUPPLEMENT_ROOT),
    'train_rows': 24_107, 'original_validation_rows': 7_861,
    'supplement_validation_rows': 194,
    'checkpoint_selection_source': 'original_gold_validation_only',
    'mini_epoch3_quality_checks': mini_checks,
    'mini_protocol_deviation': (
        'Epochs 0-3 completed and epoch 3 passed every registered gate; '
        'epoch 4 was interrupted by the Kaggle T4 quota.'
    ),
    'review_mode': review_attestation['review_mode'],
    'test_rows_read': 0,
}
if CONTRACT_PATH.exists():
    previous_contract = json.loads(CONTRACT_PATH.read_text(encoding='utf-8'))
    assert previous_contract == contract, 'Resume settings differ from the frozen run contract.'
else:
    CONTRACT_PATH.write_text(json.dumps(contract, indent=2), encoding='utf-8')
print(json.dumps(contract, indent=2))

# Prove the corrected batch/pixel profile fits before starting the long run.
PROFILE_SMOKE_PATH = RUN_DIR / 'profile_smoke_pass.json'
profile_smoke = {
    'status': 'PASS', 'git_commit': CODE_COMMIT,
    'config': CONFIG_PATH, 'min_pixels': MIN_PIXELS, 'max_pixels': MAX_PIXELS,
    'physical_batch_size': PHYSICAL_BATCH_SIZE, 'grad_accum': GRAD_ACCUM,
    'effective_batch_size': EFFECTIVE_BATCH_SIZE, 'test_rows_read': 0,
}
if PROFILE_SMOKE_PATH.exists():
    previous_smoke = json.loads(PROFILE_SMOKE_PATH.read_text(encoding='utf-8'))
    assert previous_smoke == profile_smoke, 'Existing DGX smoke belongs to another profile.'
    print('DGX PROFILE SMOKE ALREADY PASSED:', PROFILE_SMOKE_PATH)
else:
    smoke_command = [
        sys.executable, 'scripts/run_gold.py', '--stage', 'smoke',
        '--config', CONFIG_PATH, '--data-root', str(ORIGINAL_ROOT),
        '--supplement-root', str(SUPPLEMENT_ROOT), '--seed', str(SEED),
        '--num-workers', str(NUM_WORKERS),
        '--min-pixels', str(MIN_PIXELS), '--max-pixels', str(MAX_PIXELS),
    ]
    print('Running DGX profile smoke:', ' '.join(smoke_command))
    subprocess.run(smoke_command, check=True)
    PROFILE_SMOKE_PATH.write_text(json.dumps(profile_smoke, indent=2), encoding='utf-8')
    print('DGX PROFILE SMOKE PASSED:', PROFILE_SMOKE_PATH)


In [ ]:
# 7. Start once, or automatically resume the same run from last.ckpt.
if FULL_REPORT_JSON.exists():
    print('Completed report already exists; training will not run again:', FULL_REPORT_JSON)
else:
    resume_path = LAST_CHECKPOINT if AUTO_RESUME and LAST_CHECKPOINT.is_file() else None
    if resume_path is None and METRICS_CSV.exists():
        raise AssertionError(
            'Metrics exist without last.ckpt. Preserve/restore the complete seed directory '
            'or choose a new seed/run directory.'
        )
    command = [
        sys.executable, 'scripts/run_gold.py', '--stage', 'full',
        '--config', CONFIG_PATH, '--data-root', str(ORIGINAL_ROOT),
        '--supplement-root', str(SUPPLEMENT_ROOT),
        '--epochs', str(MAX_EPOCHS), '--seed', str(SEED),
        '--num-workers', str(NUM_WORKERS),
        '--min-pixels', str(MIN_PIXELS), '--max-pixels', str(MAX_PIXELS),
        '--checkpoint-every-steps', str(CHECKPOINT_EVERY_STEPS),
        '--checkpoint-root', str(CHECKPOINT_ROOT),
        '--result-csv', str(METRICS_CSV),
        '--diagnostics-json', str(DIAGNOSTICS_JSON),
        '--report-json', str(FULL_REPORT_JSON),
        '--source-validation-csv', str(SOURCE_VALIDATION_CSV),
    ]
    if resume_path is not None:
        command.extend(['--resume-checkpoint', str(resume_path)])
        print('RESUMING SAME RUN:', resume_path)
    else:
        print('STARTING FRESH FULL RUN')
    print('Running:', ' '.join(command))
    completed = subprocess.run(command, check=False)
    if completed.returncode != 0:
        print('Training stopped. Rerun this notebook with the same settings.')
        print('Expected resume checkpoint:', LAST_CHECKPOINT)
        raise RuntimeError(f'Full runner exited with code {completed.returncode}')
    assert FULL_REPORT_JSON.is_file(), 'Training exited without a final report.'


In [ ]:
# 8. Verify final artifacts and maintain one cross-model summary CSV. Test remains unopened.
import pandas as pd

report = json.loads(FULL_REPORT_JSON.read_text(encoding='utf-8'))
assert report['stage'] == 'full'
assert report['test_rows_read'] == 0
assert report['train_rows'] == 24_107
assert report['val_rows'] == 7_861
assert report['source_validation']['primary_original_gold']['checkpoint_selection_source'] is True
assert report['source_validation']['supplement_retry_abort']['rows'] == 194
assert report['checkpoint_roundtrip'] is True
assert all(path.is_file() for path in [
    METRICS_CSV, DIAGNOSTICS_JSON, FULL_REPORT_JSON, SOURCE_VALIDATION_CSV,
])
selected = next(
    row for row in report['history']
    if int(row['epoch']) == int(report['selected_epoch'])
)
summary_row = {
    'model_id': ACTIVE_MODEL_ID, 'seed': SEED, 'status': report['status'],
    'selected_epoch': report['selected_epoch'],
    'selected_checkpoint': report['best_checkpoint'],
    'checkpoint_sha256': report['selected_checkpoint_sha256'],
    **{name: selected[name] for name in [
        'outcome_mcc', 'failure_macro_f1', 'failtype_macro_f1',
        'action_macro_f1', 'needs_recovery_macro_f1',
        'strategy_attempted_macro_f1', 'recovery_outcome_mcc',
        'memory_mcc', 'bbox_mean_iou', 'bbox_recall_iou50', 'outcome_ece',
    ]},
}
MASTER_RESULTS_CSV = WORKSPACE_ROOT / 'outputs' / 'full_model_results.csv'
new_row = pd.DataFrame([summary_row])
if MASTER_RESULTS_CSV.exists():
    previous = pd.read_csv(MASTER_RESULTS_CSV)
    previous = previous[~((previous['model_id'] == ACTIVE_MODEL_ID) & (previous['seed'] == SEED))]
    new_row = pd.concat([previous, new_row], ignore_index=True)
new_row.to_csv(MASTER_RESULTS_CSV, index=False)
print(pd.DataFrame(report['history']))
print('Selected:', json.dumps(summary_row, indent=2))
print('Epoch CSV:', METRICS_CSV)
print('Cross-model CSV:', MASTER_RESULTS_CSV)
assert report['status'] == 'PASS', (
    'Full training finished, but no validation epoch passed every registered quality gate. '
    'Keep all artifacts and diagnose before opening the locked test.'
)


In [ ]:
# 9. Optional explicit backup. This runs only after you set the flag to True.
if UPLOAD_COMPLETED_RUN_TO_HF:
    from huggingface_hub import HfApi
    assert report['status'] == 'PASS'
    assert 'REPLACE_ME' not in MODEL_ARTIFACT_REPO
    api = HfApi(token=HF_TOKEN)
    api.create_repo(
        repo_id=MODEL_ARTIFACT_REPO, repo_type='model',
        private=True, exist_ok=True,
    )
    api.upload_folder(
        repo_id=MODEL_ARTIFACT_REPO, repo_type='model',
        folder_path=str(RUN_DIR),
        commit_message=f'{ACTIVE_MODEL_ID} seed {SEED} completed full run',
    )
    print('Uploaded completed run:', MODEL_ARTIFACT_REPO)
else:
    print('Artifact upload disabled. The persistent local run directory is unchanged.')


## Later models in this same notebook

Change `ACTIVE_MODEL_ID` only after the selected model has its own smoke and controlled-mini evidence. Qwen2.5-VL-3B has a compatible Gold v2.8 configuration but is intentionally blocked from full training until then. SigLIP/RoBERTa, CLIP/RoBERTa, Florence-2, InternVL2, the ablations and the baseline runners are not currently complete in this repository; the notebook will not pretend that a config name is executable evidence.

For every later model: implement the backbone, run its 16-row smoke, run its reviewed controlled mini, record the gate report, set only that registry entry to `full_authorized=True`, then rerun this notebook with a new model ID and its own persistent run directory.